# CDCN và PRNet · Chấm Test OULU-NPU Protocol 1

1. Mở notebook trên Colab và chạy cell 1–3 trên **CPU**. Cell 3 xác minh
   checkpoint 100 epoch và ngưỡng Dev, rồi chuẩn bị 600 video Test với
   MTCNN và PRNet, 8 khung mỗi video. Cell này lưu mỗi video đã xử lý lên
   Drive và có thể chạy lại nếu runtime ngắt.
2. Khi thấy `TEST_DATA_READY`, chuyển sang GPU rồi chạy lại cell 1–3 để
   khôi phục cache trong runtime mới. Cell 4 đọc checkpoint đã khóa và
   chấm Test. Notebook in đường dẫn kết quả sau `CDCN_LOCKED_TEST_READY`.

Chỉ số dùng ngưỡng từ Dev và gộp ảnh in cùng video phát lại khi tính APCER.
Thư mục kết quả chứa JSON, điểm từng video, thông tin khóa và log.

In [ ]:
#@title 1. Kết nối Drive và định vị lượt 100 epoch
from pathlib import Path
import os
import shutil
import sys
import time
import torch
from google.colab import drive

REMOUNT_DRIVE = False
drive.mount("/content/drive", force_remount=REMOUNT_DRIVE)
DATA = Path("/content/drive/MyDrive/face-pad/datasets/OULU-NPU")
BASE = Path("/content/drive/MyDrive/face-pad/cdcn-prnet-train-pilot-p1")
TRAIN_RUN = BASE / "full-runs" / "1794972293512067663" / "train-1794973542248985819"
EVAL_DIR = BASE / "locked-test-p1" / TRAIN_RUN.name
RUN = EVAL_DIR / str(time.time_ns())
WORK = Path("/content/cdcn-prnet-locked-test")
RESULT = RUN / "test"

def require_pending():
    saved = next(EVAL_DIR.glob("*/test/test_metrics.json"), None)
    if saved is not None:
        raise FileExistsError(f"Đã có kết quả Test tại {saved}")

require_pending()
if not TRAIN_RUN.is_dir():
    raise FileNotFoundError(TRAIN_RUN)
RUN.mkdir(parents=True)
print("Thư mục kết quả:", RUN)
print("Runtime:", torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU")

In [ ]:
#@title 2. Nạp code đi kèm
BUNDLE_SHA256 = '92b8bdf8c5e39b1c7709138117fba2d8633313ca30dcad361b0191354e92be2d'
BUNDLE = (
    '...'
)
import base64
import hashlib
import io
import tarfile
bundle = base64.b64decode(BUNDLE)
if hashlib.sha256(bundle).hexdigest() != BUNDLE_SHA256:
    raise ValueError("Code đi kèm không khớp SHA-256")
WORK.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(bundle), mode="r:gz") as archive:
    for member in archive:
        dest = (WORK / member.name).resolve()
        if not member.isfile() or not dest.is_relative_to(WORK):
            raise ValueError(f"Invalid bundle member: {member.name}")
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(archive.extractfile(member).read())
(RUN / f"source-{BUNDLE_SHA256[:12]}.tar.gz").write_bytes(bundle)
sys.path.insert(0, str(WORK / "src"))
from deepface_pad.progress import Progress, run_logged


# Ghi tiến trình lên ổ runtime. Chỉ chép log lên Drive sau khi lệnh kết thúc.
LOCAL_LOGS = WORK / "logs" / RUN.name

def run(label, *args):
    log = LOCAL_LOGS / f"{label}-{time.time_ns()}.log"
    try:
        run_logged(args, label=label, log=log,
                   env={**os.environ, "PYTHONPATH": str(WORK / "src"), "TF_ENABLE_ONEDNN_OPTS": "0"})
    finally:
        if log.is_file():
            try:
                shutil.copyfile(log, RUN / log.name)
            except OSError as exc:
                print(f"Chưa chép được log lên Drive: {exc} | Log cục bộ: {log}", flush=True)


In [ ]:
#@title 3. Chuẩn bị hoặc khôi phục 600 video Test
def prepare_prnet_environment():
    global PY, PRNET, RENDERER
    run("install-uv", sys.executable, "-m", "pip", "install", "uv")
    VENV = WORK / "venv"
    PY = VENV / "bin/python"
    if not PY.is_file():
        run("create-venv", sys.executable, "-m", "uv", "venv", "--python", "3.12.10", VENV)
    run("install-prnet", sys.executable, "-m", "uv", "pip", "install", "--python", PY,
        *('tensorflow==2.16.2', 'tf-keras==2.16.0', 'tf-slim==1.1.0', 'mtcnn==1.0.0', 'numpy==1.26.4', 'scikit-image==0.24.0', 'scipy==1.17.1', 'opencv-python-headless==4.11.0.86'))
    PRNET = WORK / "PRNet"
    RENDERER = WORK / "PRNet-Depth-Generation"
    for path, repo, commit in [
        (PRNET, "https://github.com/yfeng95/PRNet.git", "fc12fe5e1f1462bdea52409b213d0cf1c8cf6c5b"),
        (RENDERER, "https://github.com/clks-wzz/PRNet-Depth-Generation.git", "c6d10f2eb6ab962ab314f81ea4aa00af1d93dbbb"),
    ]:
        if not path.exists():
            run(f"clone-{path.name}", "git", "clone", repo, path)
        run(f"checkout-{path.name}", "git", "-C", path, "checkout", "--detach", commit)
    WEIGHT = PRNET / "Data/net-data/256_256_resfcn256_weight.data-00000-of-00001"
    def weight_matches(path):
        if not path.is_file():
            return False
        with path.open("rb") as stream:
            return hashlib.file_digest(stream, "sha256").hexdigest() == "59d338bf0c21b10be813f17f84042c9d3a5bf318e588855ff2b2c041c25bb91a"

    if not weight_matches(WEIGHT):
        partial = WEIGHT.with_suffix(".part")
        run("download-prnet", "curl", "-LfsS", "--retry", "2", "-o", partial,
            "https://drive.usercontent.google.com/download?id=1UoE-XuW1SDLUjZmJPkIZ1MLxvQFgmTFH&export=download&confirm=t")
        if not weight_matches(partial):
            raise ValueError("Tệp tải về không khớp SHA-256 của checkpoint PRNet")
        partial.replace(WEIGHT)

import json
from deepface_pad.colab_cache import SPLIT_ARCHIVES, copy_progress, file_hash

test_ready = False
require_pending()
package = WORK / "scripts"
sys.path.insert(0, str(package))
from cdcn_prnet_locked_test import verify_run

protocol = DATA / "Protocols.tar"
with Progress("Verify frozen model and Dev threshold"):
    locked = verify_run(TRAIN_RUN, protocol)
RESULT.mkdir(parents=True, exist_ok=True)
(RESULT / "test_lock.json").write_text(json.dumps(locked, indent=2), encoding="utf-8")
print("TEST_LOCKED", locked["epoch"], "| ngưỡng Dev:", locked["threshold_from_dev"], flush=True)

name, expected_bytes = SPLIT_ARCHIVES["test"]
source = DATA / name
if source.stat().st_size != expected_bytes:
    raise ValueError(f"Kiểm tra dung lượng Test TAR: {source}")
with source.open("rb") as stream:
    prefix_hash = hashlib.sha256(stream.read(65536)).hexdigest()
config = {"protocol": locked["protocol_sha256"], "profile": "p1-test8-mtcnn-prnet",
          "archive": {"name": name, "bytes": expected_bytes, "prefix_sha256": prefix_hash},
          "workers": {script: hashlib.sha256((package / script).read_text(encoding="utf-8").encode()).hexdigest()
                      for script in ("cdcn_prnet_smoke.py", "cdcn_prnet_expanded.py")}}
key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]
cache = BASE / "test-cache" / key
assets = WORK / f"assets-{key}"
(RUN / "cache-config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")

REPAIR_CACHE = False  # đặt True nếu đủ 600 ZIP nhưng worker báo cache hỏng
zip_count = len(list((cache / "Test").glob("*.zip")))
print("TEST_CACHE", cache, "|", zip_count, "/ 600 ZIP", flush=True)
complete = zip_count == 600 and not REPAIR_CACHE
stage = []
if complete:
    python = sys.executable
else:
    prepare_prnet_environment()
    run("install-prep-io", sys.executable, "-m", "uv", "pip", "install", "--python", PY, "pandas==2.2.3")
    python = PY
    if shutil.disk_usage(WORK).free > expected_bytes + 12 * 2**30:
        stage = ["--stage-dir", WORK / "archives"]
    else:
        print("Đọc Test TAR trực tiếp từ Drive vì ổ runtime còn ít dung lượng.", flush=True)

run("prepare-locked-test", python, "-u", package / "cdcn_prnet_expanded.py",
    "--data", DATA, "--output", assets, "--cache", cache, "--prnet", WORK / "PRNet",
    "--renderer", WORK / "PRNet-Depth-Generation", "--full", "--splits", "Test",
    *stage, *(["--restore-only"] if complete else []))
manifest = assets / "smoke_summary.json"
copy_progress(manifest, RUN / "preparation.json", "Save Test preparation manifest")
if file_hash(manifest) != file_hash(RUN / "preparation.json"):
    raise OSError("Manifest Test trên Drive không khớp bản runtime")
test_ready = True
print("TEST_DATA_READY 600 videos | 4800 frames", flush=True)

In [ ]:
#@title 4. Chấm Test với checkpoint và ngưỡng đã khóa
if not globals().get("test_ready"):
    raise RuntimeError("Chạy cell 1–3 đến TEST_DATA_READY.")
if not torch.cuda.is_available():
    raise RuntimeError("Chuyển sang runtime GPU rồi chạy lại cell 1–4.")
require_pending()
run("score-locked-test", sys.executable, "-u", package / "cdcn_prnet_locked_test.py",
    "--run", TRAIN_RUN, "--assets", assets, "--protocol", protocol,
    "--work", WORK / "scoring" / RUN.name, "--output", RESULT, "--device", "cuda")